# Brain Tumour MRI Classification - Model Comparison
Binary classification (No Tumor / Tumor). Compares a baseline CNN against ImageNet transfer-learning models
(MobileNetV2, EfficientNetB0) on a **de-duplicated, leakage-free** split, with full test-set evaluation.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Upload your `dataset` folder (containing `Training/` and `Testing/`)
to Google Drive at `MyDrive/brain_tumour/dataset`. Uploading a zip is much faster: upload `dataset.zip` to `MyDrive/brain_tumour/`
and set `USE_ZIP = True` below.

In [ ]:
import tensorflow as tf
print(tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, zipfile

USE_ZIP = False   # True if you uploaded MyDrive/brain_tumour/dataset.zip instead of the folder
if USE_ZIP:
    zipfile.ZipFile("/content/drive/MyDrive/brain_tumour/dataset.zip").extractall("/content/")
    DATA = "/content/dataset"
else:
    DATA = "/content/drive/MyDrive/brain_tumour/dataset"

OUT = "/content/drive/MyDrive/brain_tumour/results"   # metrics, plots and models are saved here
os.makedirs(OUT, exist_ok=True)
for split in ("Training", "Testing"):
    for cls in ("No Tumor", "Yes Tumor"):
        p = os.path.join(DATA, split, cls)
        print(p, len(os.listdir(p)))

In [ ]:
import hashlib, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

SEED, BATCH, VAL_FRACTION = 42, 32, 0.15
EPOCHS, FT_EPOCHS = 15, 8
IMG_SIZE = {"baseline_cnn": 128, "mobilenetv2": 224, "efficientnetb0": 224}
tf.keras.utils.set_random_seed(SEED)

## 1. Data cleaning - remove duplicates and train/test leakage
The raw dataset contains byte-identical duplicate images inside `Training`, and some `Testing` images are exact copies of
`Training` images. Evaluating on those would inflate every metric, so we remove them.

In [ ]:
def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def list_split(split):
    rows = []
    for label, cls in enumerate(("No Tumor", "Yes Tumor")):      # 0 = No Tumor, 1 = Tumor
        folder = os.path.join(DATA, split, cls)
        for fn in sorted(os.listdir(folder)):
            path = os.path.join(folder, fn)
            rows.append((path, label, md5(path)))
    return rows

train_raw, test_raw = list_split("Training"), list_split("Testing")

def dedupe(rows, banned=frozenset()):
    seen, kept = set(), []
    for path, label, h in rows:
        if h in seen or h in banned:
            continue
        seen.add(h)
        kept.append((path, label, h))
    return kept

train_all = dedupe(train_raw)
test = dedupe(test_raw, banned={h for _, _, h in train_raw})
print(f"Training: {len(train_raw)} -> {len(train_all)} after removing duplicates")
print(f"Testing : {len(test_raw)} -> {len(test)} after removing duplicates and train overlap")

train, val = train_test_split(train_all, test_size=VAL_FRACTION, random_state=SEED, stratify=[r[1] for r in train_all])
data_stats = {"train_raw": len(train_raw), "train_dedup": len(train_all), "test_raw": len(test_raw),
              "test_clean": len(test), "train": len(train), "val": len(val)}
print(data_stats)
print("Class balance (train):", np.bincount([r[1] for r in train]), "| (test):", np.bincount([r[1] for r in test]))

In [ ]:
# Look at a few samples
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
samples = [r for r in train if r[1] == 0][:6] + [r for r in train if r[1] == 1][:6]
for ax, (path, label, _) in zip(axes.ravel(), samples):
    ax.imshow(plt.imread(path), cmap="gray")
    ax.set_title("Tumor" if label else "No Tumor")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Input pipeline

In [ ]:
AUTO = tf.data.AUTOTUNE

def make_ds(rows, size, shuffle=False):
    def decode(path, label):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.resize(img, (size, size))
        return tf.cast(img, tf.float32) / 255.0, tf.cast(label, tf.float32)[None]   # images scaled to 0-1
    ds = tf.data.Dataset.from_tensor_slices(([r[0] for r in rows], [r[1] for r in rows])).map(decode, AUTO).cache()
    if shuffle:
        ds = ds.shuffle(len(rows), seed=SEED)
    return ds.batch(BATCH).prefetch(AUTO)

## 3. Models
All models take **0-1 scaled** images (so the backend's `/255` preprocessing works unchanged) and output a sigmoid tumour probability.

In [ ]:
def augment():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
    ], name="augment")

def build_baseline(size):
    m = tf.keras.Sequential([
        tf.keras.Input((size, size, 3)), augment(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(), tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name="baseline_cnn")
    return m, None

def build_transfer(name, size):
    if name == "mobilenetv2":
        base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(size, size, 3))
        prep = tf.keras.layers.Rescaling(2.0, offset=-1.0)     # 0-1 -> -1..1
    else:
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(size, size, 3))
        prep = tf.keras.layers.Rescaling(255.0)                # EfficientNet expects 0-255
    base.trainable = False
    inp = tf.keras.Input((size, size, 3))
    x = prep(augment()(inp))
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name=name), base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

def fit(model, train_ds, val_ds, epochs, history):
    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cb, verbose=2)
    for k, v in h.history.items():
        history.setdefault(k, []).extend(float(i) for i in v)

## 4. Evaluation helpers

In [ ]:
def evaluate(model, test_ds):
    y_true = np.concatenate([y.numpy().ravel() for _, y in test_ds]).astype(int)
    prob = model.predict(test_ds, verbose=0).ravel()
    pred = (prob > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "accuracy": float((tp + tn) / len(y_true)),
        "precision": float(precision_score(y_true, pred)),
        "recall_sensitivity": float(recall_score(y_true, pred)),
        "specificity": float(tn / (tn + fp)),
        "f1": float(f1_score(y_true, pred)),
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "n_test": int(len(y_true)),
    }, y_true, prob

def plot_curves(name, h):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="val"); ax[0].set_title(f"{name} loss")
    ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="val"); ax[1].set_title(f"{name} accuracy")
    for a in ax:
        a.set_xlabel("epoch")
        a.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT}/curves_{name}.png", dpi=120)
    plt.show()

def plot_cm(name, cm):
    arr = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(arr, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, arr[i, j], ha="center", va="center", color="white" if arr[i, j] > arr.max() / 2 else "black")
    ax.set_xticks([0, 1], ["No Tumor", "Tumor"])
    ax.set_yticks([0, 1], ["No Tumor", "Tumor"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
    plt.tight_layout()
    plt.savefig(f"{OUT}/cm_{name}.png", dpi=120)
    plt.show()

## 5. Train and evaluate
Transfer-learning models are trained in two stages: (1) train the new classification head with the backbone frozen,
(2) fine-tune the top 30 backbone layers at a low learning rate (BatchNorm kept frozen).
The **validation** set is used for early stopping; the **test** set is only used once, for the final numbers.

In [ ]:
results = {"models": {}, "data": data_stats}
rocs = {}

for name in ["baseline_cnn", "mobilenetv2", "efficientnetb0"]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    size = IMG_SIZE[name]
    train_ds, val_ds, test_ds = make_ds(train, size, True), make_ds(val, size), make_ds(test, size)
    history, t0 = {}, time.time()

    model, base = build_baseline(size) if name == "baseline_cnn" else build_transfer(name, size)
    compile_model(model, 1e-3)
    fit(model, train_ds, val_ds, EPOCHS, history)

    if base is not None:
        base.trainable = True
        for layer in base.layers[:-30]:
            layer.trainable = False
        for layer in base.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        compile_model(model, 1e-5)
        fit(model, train_ds, val_ds, FT_EPOCHS, history)

    model.save(f"{OUT}/{name}.keras")
    m, y_true, prob = evaluate(model, test_ds)
    m.update(val_accuracy=float(max(history["val_accuracy"])), params=int(model.count_params()),
             input_size=size, train_seconds=round(time.time() - t0, 1), epochs_run=len(history["loss"]))
    results["models"][name] = {**m, "history": history}
    rocs[name] = (y_true, prob)
    plot_curves(name, history)
    plot_cm(name, m["confusion_matrix"])
    print({k: round(v, 4) for k, v in m.items() if isinstance(v, float)})
    tf.keras.backend.clear_session()

json.dump(results, open(f"{OUT}/metrics.json", "w"), indent=2)

## 6. Results summary

In [ ]:
cols = ["accuracy", "precision", "recall_sensitivity", "specificity", "f1", "roc_auc", "val_accuracy", "params", "train_seconds"]
summary = pd.DataFrame({n: {c: m[c] for c in cols} for n, m in results["models"].items()}).T
display(summary.style.format({c: "{:.4f}" for c in cols[:7]}).highlight_max(subset=cols[:7], color="#bbf7d0"))
summary.to_csv(f"{OUT}/summary.csv")

fig, ax = plt.subplots(figsize=(5, 4.5))
for name, (yt, pr) in rocs.items():
    fpr, tpr, _ = roc_curve(yt, pr)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(yt, pr):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves (test set)"); ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/roc.png", dpi=120)
plt.show()

## 7. Error analysis - which scans does the best model get wrong?
In tumour screening a **false negative** (missed tumour) is the costly error, so check sensitivity and look at the misses.

In [ ]:
best = max(results["models"], key=lambda n: results["models"][n]["f1"])
print("Best model by test F1:", best)
results["best_model"] = best
json.dump(results, open(f"{OUT}/metrics.json", "w"), indent=2)

model = tf.keras.models.load_model(f"{OUT}/{best}.keras")
size = IMG_SIZE[best]
prob = model.predict(make_ds(test, size), verbose=0).ravel()
y = np.array([r[1] for r in test])
wrong = np.where((prob > 0.5).astype(int) != y)[0]
print(f"{len(wrong)} misclassified of {len(test)}")
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, i in zip(axes.ravel(), wrong[:12]):
    ax.imshow(plt.imread(test[i][0]), cmap="gray")
    ax.set_title(f"true={'T' if y[i] else 'N'}  p={prob[i]:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/misclassified.png", dpi=120)
plt.show()

## 8. Grad-CAM (explainability) on the best model
Highlights the image regions that pushed the model towards "tumour". Use this to check the model looks at the lesion and not at
background artefacts.

In [ ]:
import cv2

def gradcam_heatmap(model, batch):
    backbone = next((l for l in model.layers if isinstance(l, tf.keras.Model) and not isinstance(l, tf.keras.Sequential)), None)
    x = tf.constant(batch)
    if backbone is not None:
        # transfer model layout: [input, augment, rescaling, backbone, pooling, dropout, dense]
        i = model.layers.index(backbone)
        pre, head = model.layers[i - 1], model.layers[i + 1:]
        with tf.GradientTape() as tape:
            feats = backbone(pre(x), training=False)
            tape.watch(feats)
            z = feats
            for l in head:
                z = l(z, training=False)
            score = z[:, 0]
        grads = tape.gradient(score, feats)
    else:
        last = [l for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)][-1]
        gm = tf.keras.Model(model.inputs, [last.output, model.output])
        with tf.GradientTape() as tape:
            feats, out = gm(x, training=False)
            score = out[:, 0]
        grads = tape.gradient(score, feats)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * feats, axis=-1))[0].numpy()
    return cam / (cam.max() + 1e-8)

hits = [i for i in range(len(test)) if y[i] == 1 and prob[i] > 0.5][:6]
fig, axes = plt.subplots(2, len(hits), figsize=(2.3 * len(hits), 5))
for k, i in enumerate(hits):
    img = cv2.cvtColor(cv2.imread(test[i][0]), cv2.COLOR_BGR2RGB)
    batch = (cv2.resize(img, (size, size)) / 255.0).astype("float32")[None]
    cam = cv2.resize(gradcam_heatmap(model, batch), (img.shape[1], img.shape[0]))
    heat = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    axes[0, k].imshow(img); axes[0, k].axis("off")
    axes[1, k].imshow(cv2.addWeighted(img, 0.5, heat, 0.5, 0)); axes[1, k].axis("off")
plt.suptitle(f"Grad-CAM - {best}")
plt.tight_layout()
plt.savefig(f"{OUT}/gradcam_examples.png", dpi=120)
plt.show()

## 9. Download results
Everything is already saved in `MyDrive/brain_tumour/results/`. Copy these into your local project:
- `metrics.json` -> `brain_tumour/ml/results/metrics.json`
- `<best>.keras` -> `brain_tumour/model/`
- the `.png` plots -> `brain_tumour/ml/results/` (for your report)

Then add `"deployed_model": "<best>"` to `metrics.json` so the backend loads that model.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("/content/results", "zip", OUT)
files.download("/content/results.zip")